In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split,GridSearchCV,StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,ConfusionMatrixDisplay)

# Cargar el CSV original
df = pd.read_csv("online_retail_II.csv")

Saving online_retail_II.csv to online_retail_II (1).csv


In [ ]:
# Homologar los nombres de las columnas con los que usaremos en el notebook.
# El CSV tiene "Customer ID", "Invoice" y "Price".
df = df.rename(columns={
    "Customer ID": "CustomerID",
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice"
})

# Preparación básica
df = df.dropna(subset=["CustomerID", "Description"]).copy()
df["CustomerID"] = df["CustomerID"].astype("int64")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

# Excluir cancelaciones/devoluciones y precios no válidos
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()

# Fecha de corte: solo el período anterior crea las variables predictoras
cutoff_date = pd.Timestamp("2011-06-01")

df_hist = df[df["InvoiceDate"] < cutoff_date].copy()
df_futuro = df[df["InvoiceDate"] >= cutoff_date].copy()

# Importe de cada renglón, calculado con el período histórico
df_hist["Total_Line"] = df_hist["Quantity"] * df_hist["UnitPrice"]

# Variables históricas agregadas por cliente
features = (
    df_hist.groupby("CustomerID")
    .agg(
        Num_Invoices=("InvoiceNo", "nunique"),
        Total_Spent=("Total_Line", "sum")
    )
)

# Tamaño promedio de la canasta:
# primero suma las unidades de cada factura y luego promedia por cliente
basket_size = (
    df_hist.groupby(["CustomerID", "InvoiceNo"])["Quantity"]
    .sum()
    .groupby("CustomerID")
    .mean()
    .rename("Avg_Basket_Size")
)

features = features.join(basket_size)

# Target: 1 si el cliente vuelve a comprar después del corte; 0 si no
customers_future = set(df_futuro["CustomerID"].unique())
features["Will_Return"] = features.index.isin(customers_future).astype(int)

# Matriz de variables y variable objetivo
columnas_x = ["Num_Invoices", "Total_Spent", "Avg_Basket_Size"]
X = features[columnas_x]
y = features["Will_Return"]

print("Distribución del target (0 = no regresa, 1 = regresa):")
print(y.value_counts(normalize=True).sort_index())


Distribución del target (0 = no regresa, 1 = regresa):
Will_Return
0    0.472532
1    0.527468
Name: proportion, dtype: float64


In [ ]:
# Partición 80/20, reproducible y estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Configurar GridSearchCV para afinar el Random Forest (Tuning)
# Buscamos la mejor combinación limitando profundidad y tamaño de división para evitar overfitting
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [3, 4, 5, 6, 7, 8, 9, 10],
    'min_samples_split': [5, 10, 20, 50],
    'class_weight': ['balanced', None]
}

rf_base = RandomForestClassifier(random_state=42)

# Validación cruzada estratificada de 5 particiones
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

# 2. Entrenar el modelo con el conjunto de Train escalado
grid_search.fit(X_train_scaled, y_train)

# Obtener el mejor estimador optimizado
print("Mejores hiperparámetros")
print(grid_search.best_params_)

Mejores hiperparámetros
{'class_weight': None, 'max_depth': 4, 'min_samples_split': 20, 'n_estimators': 100}


In [ ]:
# ==========================
# 10 EXPERIMENTOS
# ==========================

resultados = []
roc_auc_experimentos = []
mejor_auc = 0

for experimento in range(1, 11):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=experimento
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=4,
        min_samples_split=20,
        class_weight=None,
        random_state=experimento
    )

    rf.fit(X_train_scaled,y_train)

    y_pred = rf.predict(X_test_scaled)

    y_prob = rf.predict_proba(X_test_scaled)[:,1]

    accuracy = accuracy_score(y_test,y_pred)

    precision = precision_score(y_test,y_pred,zero_division=0)

    recall = recall_score(y_test,y_pred,zero_division=0)

    f1 = f1_score(y_test,y_pred,zero_division=0)

    auc = roc_auc_score(y_test,y_prob)

    roc_auc_experimentos.append(auc)

    resultados.append([
        accuracy,
        precision,
        recall,
        f1,
        auc])

    if auc > mejor_auc:

        mejor_auc = auc

        mejor_modelo = rf

        mejor_y_test = y_test.copy()
        mejor_y_pred = y_pred.copy()

resultados = np.array(resultados)

tabla_roc = pd.DataFrame({"Experimento": range(1, 11),"ROC-AUC": np.round(roc_auc_experimentos, 4)})

print("ROC-AUC por experimento")
display(tabla_roc)

print("ROC-AUC Promedio:",round(np.mean(roc_auc_experimentos), 4))

tabla_experimentos = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC"
    ],
    "Promedio": [
        resultados[:,0].mean(),
        resultados[:,1].mean(),
        resultados[:,2].mean(),
        resultados[:,3].mean(),
        resultados[:,4].mean()
    ],
    "Desv. Estándar": [
        resultados[:,0].std(),
        resultados[:,1].std(),
        resultados[:,2].std(),
        resultados[:,3].std(),
        resultados[:,4].std()
    ]
})

print("\nRESULTADOS PROMEDIO DE 10 EXPERIMENTOS")
display(
    tabla_experimentos.round(4)
)




ROC-AUC por experimento


,Experimento,ROC-AUC
0,1,0.7877
1,2,0.7768
2,3,0.7732
3,4,0.7904
4,5,0.7795
5,6,0.7701
6,7,0.7739
7,8,0.7841
8,9,0.7721
9,10,0.7495


ROC-AUC Promedio: 0.7757

RESULTADOS PROMEDIO DE 10 EXPERIMENTOS


,Métrica,Promedio,Desv. Estándar
0,Accuracy,0.7040,0.0123
1,Precision,0.7311,0.0156
2,Recall,0.6954,0.0195
3,F1-Score,0.7126,0.0122
4,ROC-AUC,0.7757,0.0109


In [ ]:
# 4. Gráfica de Feature Importance para el mejor Random Forest
importances = (mejor_modelo.feature_importances_)

feature_imp_df = pd.DataFrame({
    "Feature": columnas_x,
    "Importance": importances
}).sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(8, 5))
plt.barh(feature_imp_df['Feature'], feature_imp_df['Importance'], color='#3470a3')
plt.title("Importancia de Variables")
plt.xlabel("Importancia")
plt.show()

In [ ]:
cm_rf = confusion_matrix(mejor_y_test,mejor_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf,display_labels=["No regresa", "Regresa"])
disp.plot(cmap="Greens")
plt.title("Matriz de Confusión - Random Forest")
plt.show()

In [ ]:

#GRAFICA DE BARRAS
tn, fp, fn, tp = cm_rf.ravel()
valores = [tn,fp,fn,tp]

etiquetas = [
    "TN\nNo regresa correcto",
    "FP\nFalsa alarma",
    "FN\nNo detectado",
    "TP\nRegresa correcto"]

plt.figure(figsize=(8,5))

plt.bar(etiquetas,valores,color=[
        "#2E8B57",
        "#FF8C00",
        "#DC143C",
        "#4682B4"])

for i, v in enumerate(valores):
    plt.text(i,v + 5,str(v),ha="center")

plt.ylabel("Clientes")
plt.title("Resultados de Clasificación")
plt.show()